## Step 1: Import Packages

In [21]:
# TensorFlow is my main library — everything runs through it
import tensorflow as tf

# I’ll build the model using Keras layers and model APIs
from keras import layers, models

## Step 2: Load the Image Datasets

In [22]:
# Load training data from the pre-split folder
from keras.utils import image_dataset_from_directory
train_dir = "../data/split/train"
val_dir = "../data/split/val"
test_dir = "../data/split/test"

train_ds = image_dataset_from_directory(
    train_dir,
    image_size=(224, 224),  # resizing images for MobileNetV2
    batch_size=32
)

# Load validation data
val_ds = image_dataset_from_directory(
     val_dir,
    image_size=(224, 224),
    batch_size=32
)

# Load test data
test_ds = image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=32
)

Found 38791 files belonging to 39 classes.
Found 8318 files belonging to 39 classes.
Found 8339 files belonging to 39 classes.


## Step 3: Preprocess with MobileNetV2

In [23]:
# I am using MobileNetV2, so I need to preprocess the images in the exact format it expects.
from keras.applications.mobilenet_v2 import preprocess_input

# I will set AUTOTUNE to let TensorFlow optimize the data pipeline behind the scenes.
AUTOTUNE = tf.data.AUTOTUNE

# Here I define a function that will apply the MobileNetV2 preprocessing to each image.
def preprocess_ds(image, label):
    image = preprocess_input(image)
    return image, label

# Now I map the preprocessing function and add caching, shuffling and prefetching for performance
train_ds = train_ds.map(preprocess_ds, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.map(preprocess_ds, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

test_ds = test_ds.map(preprocess_ds, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Step 4: Load the Pretrained MobileNetV2 Base Model

In [24]:
# I will import the base MobileNetV2 model from keras.applications
from keras.applications import MobileNetV2

# I’ll load MobileNetV2 without the top (classification) layers
# input_shape must match the shape of images in my dataset
# weights="imagenet" tells it to load the pretrained weights
# include_top=False means I don't want the final Dense layers from ImageNet classification
base_model = MobileNetV2(input_shape=(224, 224, 3),
                         include_top=False,
                         weights="imagenet")

# I will freeze the base model — I don’t want to retrain its weights
base_model.trainable = False

# I can check the model summary to understand its architecture
base_model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 9s 1us/step


Model: "mobilenetv2_1.00_224"

┏━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┓
┃ Layer       ┃ Output     ┃ Param ┃ Connected  ┃
┃ (type)      ┃ Shape      ┃     # ┃ to         ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━┩
│ input_layer │ (None,     │     0 │ -          │
│ (InputLaye… │ 224, 224,  │       │            │
│             │ 3)         │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ Conv1       │ (None,     │   864 │ input_lay… │
│ (Conv2D)    │ 112, 112,  │       │            │
│             │ 32)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ bn_Conv1    │ (None,     │   128 │ Conv1[0][… │
│ (BatchNorm… │ 112, 112,  │       │            │
│             │ 32)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ Conv1_relu  │ (None,     │     0 │ bn_Conv1[… │
│ (ReLU)      │ 112, 112,  │       │            │
│             │ 32)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ expanded_c… │ (None,     │   288 │ Conv1_rel… │
│ (Depthwise… │ 112, 112,  │       │            │
│             │ 32)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ expanded_c… │ (None,     │   128 │ expanded_… │
│ (BatchNorm… │ 112, 112,  │       │            │
│             │ 32)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ expanded_c… │ (None,     │     0 │ expanded_… │
│ (ReLU)      │ 112, 112,  │       │            │
│             │ 32)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ expanded_c… │ (None,     │   512 │ expanded_… │
│ (Conv2D)    │ 112, 112,  │       │            │
│             │ 16)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ expanded_c… │ (None,     │    64 │ expanded_… │
│ (BatchNorm… │ 112, 112,  │       │            │
│             │ 16)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_ex… │ (None,     │ 1,536 │ expanded_… │
│ (Conv2D)    │ 112, 112,  │       │            │
│             │ 96)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_ex… │ (None,     │   384 │ block_1_e… │
│ (BatchNorm… │ 112, 112,  │       │            │
│             │ 96)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_ex… │ (None,     │     0 │ block_1_e… │
│ (ReLU)      │ 112, 112,  │       │            │
│             │ 96)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_pad │ (None,     │     0 │ block_1_e… │
│ (ZeroPaddi… │ 113, 113,  │       │            │
│             │ 96)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_de… │ (None, 56, │   864 │ block_1_p… │
│ (Depthwise… │ 56, 96)    │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_de… │ (None, 56, │   384 │ block_1_d… │
│ (BatchNorm… │ 56, 96)    │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_de… │ (None, 56, │     0 │ block_1_d… │
│ (ReLU)      │ 56, 96)    │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_pr… │ (None, 56, │ 2,304 │ block_1_d… │
│ (Conv2D)    │ 56, 24)    │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_pr… │ (None, 56, │    96 │ block_1_p… │
│ (BatchNorm… │ 56, 24)    │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_2_ex… │ (None, 56, │ 3,456 │ block_1_p… │
│ (Conv2D)    │ 56, 144)   │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_2_ex… │ (None, 56, │   576 │ block_2_e… │
│ (BatchNorm… │ 56, 144)   │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_2_ex… │ (None, 56, │     0 │ block_2_e… │
│ (ReLU)      │ 56, 144)   │       │            │
├─────────────┼────────────┼───────┼────────────┤


 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)